## Fine-tuning by PyTorch

前節では`transformers.Trainer`を使ってファインチューニングしていましたが、より処理の詳細が分かりやすいようPyTorchを使ってTransformersの学習ループを実装する方法を見ていきます。

まず`transformers.Trainer`を使う場合と同様に、Transformersのデータセット、トークナイザ、collatorを読み込みます

In [ ]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding
from transformers import AutoModelForSequenceClassification

# Dataset
raw_datasets = load_dataset("nyu-mll/glue", "mrpc")

# Tokenizer
checkpoint = "bert-base-uncased"
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True,
                                      remove_columns=["sentence1", "sentence2", "idx"])
print(tokenized_datasets)

# collator
data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

# Model
model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)

なお、`raw_datasets.map`メソッドに`remove_columns=["sentence1", "sentence2", "idx"]`という引数が加わっていますが、これは"sentence1","sentence2"列は文字列型でPyTorchのDataLoaderが読み込めないので、DataLoaderに渡すタイミングで削除する処理を加えています。

このままだとPyTorchでうまく動作しないため、データセットに以下の処理を加えます

- 正解ラベルのキー名が"label"だが、Hugging Faceのモデルは"labels"にしないと自動で訓練時のLoss計算してくれないので、修正
- `.set_format("torch")`を指定することで、データを取り出した瞬間に自動的に`torch.Tensor`型に一発変換されるようにする

In [ ]:
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

サブセット（train, validation）ごとにPyTorchのDataLoaderを作成し、データセットとcollatorを渡します

In [ ]:
from torch.utils.data import DataLoader

# Create DataLoaders
train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=8, collate_fn=data_collator
)
eval_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=8, collate_fn=data_collator
)

# Check the shape of the first batch
for batch in train_dataloader:
    break
print({k: v.shape for k, v in batch.items()})

# Make sure that the model works properly with the batch
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

PyTorchでの学習に必要な、最適化計算によるパラメータ更新を行うoptimizerと、学習率をepochの進展に応じて学習率スケジューラ（learning rate scheduler）を作成します。
学習率スケジューラはPyTorch公式の`torch.optim.lr_scheduler`内のクラスを使用しても良いですが、ここでは`transformers.get_scheduler`関数で作成します。

In [ ]:
from torch.optim import AdamW
from transformers import get_scheduler

# Optimizer
optimizer = AdamW(model.parameters(), lr=5e-5)

# Learning rate scheduler
num_epochs = 3
num_training_steps = num_epochs * len(train_dataloader)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)
print(num_training_steps)

あとはPyTorchのルールに則り、学習ループと評価を実装します（`metric.add_batch`は、ミニバッチ内で評価指標計算に必要なデータのみを集約するメソッドです）

In [ ]:
import torch
from tqdm.auto import tqdm
import evaluate

# Load the model onto the GPU if available
device = torch.device("cuda") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

progress_bar = tqdm(range(num_training_steps))

# Training loop
model.train()
for epoch in range(num_epochs):
    for batch in train_dataloader:
        batch = {k: v.to(device) for k, v in batch.items()}
        outputs = model(**batch)
        loss = outputs.loss
        loss.backward()

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

# Evaluation
metric = evaluate.load("glue", "mrpc")
model.eval()
for batch in eval_dataloader:
    batch = {k: v.to(device) for k, v in batch.items()}
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    # Add batch predictions and references to the metric
    metric.add_batch(predictions=predictions, references=batch["labels"])

metric.compute()

## AccelerateによるマルチGPUでの分散トレーニング

PyTorchで分散トレーニングを行うには、DDP等の複雑な実装を適切にハンドリングする必要があります。
Acceleratorはこのような分散トレーニングを、シンプルな実装で実現できるライブラリです。

Acceleratorでは、インスタンスの`prepare()`メソッドにモデル、Optimizer、データローダ（trainとvalidation）を入れて取り出すことで、現在のインフラ（GPUが何枚あるか、TPUかなど）を自動検知し、裏側で自動的にデータを各GPUに分配するラッパーを被せ、デバイスへのデータ転送`.to("cuda")`も自動化してくれます。これによりデバイスが変わった際（例：オンプレGPU→クラウドGPU）にもコードの変更を最小限にして対応できます。

例えばここまで実施した学習ループをAcceleratorで分散トレーニングしたい場合、以下のように実装します

In [ ]:
from accelerate import Accelerator
from torch.optim import AdamW
from transformers import AutoModelForSequenceClassification, get_scheduler

accelerator = Accelerator()

model = AutoModelForSequenceClassification.from_pretrained(checkpoint, num_labels=2)
optimizer = AdamW(model.parameters(), lr=3e-5)

train_dl, eval_dl, model, optimizer = accelerator.prepare(
    train_dataloader, eval_dataloader, model, optimizer
)

num_epochs = 3
num_training_steps = num_epochs * len(train_dl)
lr_scheduler = get_scheduler(
    "linear",
    optimizer=optimizer,
    num_warmup_steps=0,
    num_training_steps=num_training_steps,
)

progress_bar = tqdm(range(num_training_steps))

# Training loop
model.train()
for epoch in range(num_epochs):
    for batch in train_dl:
        outputs = model(**batch)
        loss = outputs.loss
        accelerator.backward(loss)

        optimizer.step()
        lr_scheduler.step()
        optimizer.zero_grad()
        progress_bar.update(1)

# Evaluation
metric = evaluate.load("glue", "mrpc")
model.eval()
for batch in eval_dl:
    with torch.no_grad():
        outputs = model(**batch)

    logits = outputs.logits
    predictions = torch.argmax(logits, dim=-1)
    # Add batch predictions and references to the metric
    metric.add_batch(predictions=predictions, references=batch["labels"])

metric.compute()

実は`transformers.Trainer`クラスにもマルチGPU最適化機能がありますが、ブラックボックス性が高いため、Acceleratorはより自由度の高いPyTorchに近い学習ループをマルチGPUで実行したい際に便利です。

また`accelerate config`というコマンドを打つことで、accelerateの環境設定（使用するGPUの制限や混合精度学習の有無等）をコマンドラインで対話形式で行えます。